In [1]:
! pip install gradio scikit-learn pandas numpy matplotlib

# Learn about the Gradio : https://www.gradio.app/



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re # regular expressions modules
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB


In [4]:
data = {
    "review": [
        "I loved this movie, it was fantastic!",
        "Worst movie ever, totally boring",
        "Amazing acting and great storyline",
        "I did not like the film at all",
        "Best movie of the year",
        "Terrible plot and bad acting",
        "Absolutely wonderful experience",
        "Waste of time and money",
        "Brilliant direction and screenplay",
        "Horrible movie"
    ],
    "sentiment": [1,0,1,0,1,0,1,0,1,0]
}

df = pd.DataFrame(data)


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['review'] = df['review'].apply(clean_text)

In [6]:
# Logistic Regression (ML Pipeline)
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("classifier", LogisticRegression())
])


In [7]:
# Naive Bayes Pipeline
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("classifier", MultinomialNB())
])


In [8]:
# train both of the pipline
lr_pipeline.fit(df['review'], df['sentiment'])
nb_pipeline.fit(df['review'], df['sentiment'])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [12]:
# Core Prediction
def predict_sentiment(text, model_choice):
    text = clean_text(text)

    model = lr_pipeline if model_choice == "Logistic Regression" else nb_pipeline

    prediction = model.predict([text])[0]
    probabilities = model.predict_proba([text])[0]

    sentiment = "Positive 😊" if prediction == 1 else "Negative 😞"
    confidence = round(np.max(probabilities) * 100, 2)

    # Visualization
    fig, ax = plt.subplots()
    ax.bar(["Negative", "Positive"], probabilities)
    ax.set_ylim(0, 1)
    ax.set_title("Sentiment Probability Distribution")

    return sentiment, f"{confidence}%", fig


In [14]:
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=[
        gr.Textbox(
            lines=4,
            placeholder="Type a movie review, tweet, or feedback..."
        ),
        gr.Radio(
            ["Logistic Regression", "Naive Bayes"],
            label="Choose Model",
            value="Logistic Regression"
        )
    ],
    outputs=[
        gr.Textbox(label="Predicted Sentiment"),
        gr.Textbox(label="Confidence Score"),
        gr.Plot(label="Sentiment Probability")
    ],
    title="Advanced Sentiment Analysis NLP App",
    description="""
    This application performs real-time sentiment analysis using NLP.

    • TF-IDF Vectorization
    • Machine Learning Models
    • Confidence & Probability Visualization
    """,
    theme="default"
)


c:\Users\soura\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [15]:
interface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv
